In [0]:
from pyspark.sql import Row

In [0]:
# Create Pipeline Config Data
pipeline_configs = [

    Row(
        pipeline_name="online_retail_bronze",
        source_name="Online Retail.csv",
        source_type="csv",
        target_table="workspace.default.bronze_online_retail",
        output_type=None,
        layer="bronze",
        is_active=True
    ),

    Row(
        pipeline_name="online_retail_silver",
        source_name="workspace.default.bronze_online_retail",
        source_type="delta",
        target_table="workspace.default.silver_online_retail",
        output_type=None,
        layer="silver",
        is_active=True
    ),

    Row(
        pipeline_name="online_retail_gold",
        source_name="workspace.default.silver_online_retail",
        source_type="delta",
        target_table="workspace.default.gold_daily_sales",
        output_type="daily",
        layer="gold",
        is_active=True
    ),

    Row(
        pipeline_name="online_retail_gold",
        source_name="workspace.default.silver_online_retail",
        source_type="delta",
        target_table="workspace.default.gold_country_sales",
        output_type="country",
        layer="gold",
        is_active=True
    ),

    Row(
        pipeline_name="online_retail_gold",
        source_name="workspace.default.silver_online_retail",
        source_type="delta",
        target_table="workspace.default.gold_customer_sales",
        output_type="customer",
        layer="gold",
        is_active=True
    )

]

In [0]:
# Create DataFrame
config_df = spark.createDataFrame(
    pipeline_configs
)


In [0]:
# Save Metadata Table
config_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.pipeline_config"
    )

In [0]:
# Creating validation rules data
validation_rules = [

    Row(
        pipeline_name="online_retail_silver",
        rule_name="positive_quantity",
        rule_expression="Quantity > 0",
        reject_reason="Invalid Quantity",
        is_active=True
    ),

    Row(
        pipeline_name="online_retail_silver",
        rule_name="valid_description",
        rule_expression="Description IS NOT NULL",
        reject_reason="Missing Description",
        is_active=True
    ),

    Row(
        pipeline_name="online_retail_silver",
        rule_name="not_cancelled",
        rule_expression="InvoiceNo NOT LIKE 'C%'",
        reject_reason="Cancelled Order",
        is_active=True
    )

]

In [0]:
# Creating rules dataframe
rules_df = spark.createDataFrame(
    validation_rules
)

display(rules_df)

pipeline_name,rule_name,rule_expression,reject_reason,is_active
online_retail_silver,positive_quantity,Quantity > 0,Invalid Quantity,true
online_retail_silver,valid_description,Description IS NOT NULL,Missing Description,true
online_retail_silver,not_cancelled,InvoiceNo NOT LIKE 'C%',Cancelled Order,true


In [0]:
rules_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.validation_rules"
    )

In [0]:
display(
    spark.table(
        "workspace.default.validation_rules"
    )
)

pipeline_name,rule_name,rule_expression,reject_reason,is_active
online_retail_silver,positive_quantity,Quantity > 0,Invalid Quantity,true
online_retail_silver,valid_description,Description IS NOT NULL,Missing Description,true
online_retail_silver,not_cancelled,InvoiceNo NOT LIKE 'C%',Cancelled Order,true


In [0]:
# Create reject table
from pyspark.sql.types import *

schema = StructType([
    StructField("pipeline_name", StringType(), True),
    StructField("rule_name", StringType(), True),
    StructField("reject_reason", StringType(), True),
    StructField("rejected_timestamp", TimestampType(), True)
])

empty_df = spark.createDataFrame([], schema)

empty_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.rejected_records"
    )

In [0]:
# Dependencies data
dependencies = [

    Row(
        pipeline_name="online_retail_silver",
        depends_on="online_retail_bronze"
    ),

    Row(
        pipeline_name="online_retail_gold",
        depends_on="online_retail_silver"
    )

]

In [0]:
# write to table
dependency_df = spark.createDataFrame(
    dependencies
)

display(dependency_df)

pipeline_name,depends_on
online_retail_silver,online_retail_bronze
online_retail_gold,online_retail_silver


In [0]:
dependency_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.pipeline_dependencies"
    )